# Vector Store Utilities

The `utils.py` module contains internal numerical utilities used by LangChain's in-memory vector-store implementation.

The module is explicitly marked as a private API and may change without notice. Its only non-underscore function calculates maximal marginal relevance for a query embedding and a collection of candidate embeddings.

### Functions

1. `maximal_marginal_relevance`: Selects embedding indices using maximal marginal relevance.

   The function first selects the candidate with the highest cosine similarity to the query. It then repeatedly selects the candidate that best balances query similarity against redundancy with embeddings that have already been selected.

   `lambda_mult` controls this balance:

   - A value closer to `1` gives greater weight to query similarity.
   - A value closer to `0` gives greater weight to diversity.

   The query embedding may be one-dimensional or two-dimensional. A one-dimensional query is expanded to a single-row matrix before similarity calculation.

   The number of returned indices is limited to the smaller of `k` and the number of candidate embeddings. An empty list is returned when `k` is non-positive or `embedding_list` is empty.

   An `ImportError` is raised when NumPy is unavailable. Shape, NaN, and infinity handling during cosine-similarity calculation is delegated to the module's private `_cosine_similarity` helper.

   * **Syntax:**
     ```python
     maximal_marginal_relevance(
         query_embedding: npt.NDArray[
             np.floating
         ], # Query embedding vector or single-row matrix
         embedding_list: list[
             list[float]
         ], # Candidate embedding vectors
         lambda_mult: float = 0.5, # Query-similarity versus diversity weighting
         k: int = 4 # Maximum number of indices to return
     ) -> list[int]
     ```

## Selection Formula

For each unselected candidate, the function calculates a score equivalent to:

```text
lambda_mult × similarity_to_query
- (1 - lambda_mult) × maximum_similarity_to_selected
```

The candidate with the highest score is appended to the selected-index list during each iteration.

## Internal Components Omitted

The following implementation details are private and are not documented as public API entries:

- `_cosine_similarity`
- `_HAS_NUMPY`
- `_HAS_SIMSIMD`
- `Matrix`
- `logger`